# Players — Save, Load & Inspect

This notebook demonstrates how to create, save, and load MCTS players.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pymcts.core.players import MCTSPlayer, GreedyMCTSPlayer
from pymcts.core.config import MCTSConfig
from pymcts.arena import batched_arena
from pymcts.games.bridgit.neural_net import BridgitNet
from pymcts.games.bridgit.config import BoardConfig, NeuralNetConfig
from pymcts.games.bridgit.game import BridgitGame

## 1. Create a player from scratch

In [ ]:
board_config = BoardConfig(size=5)
net = BridgitNet(board_config=board_config, net_config=NeuralNetConfig(num_channels=32, num_res_blocks=4))
mcts_config = MCTSConfig(num_simulations=50, num_parallel_leaves=10)

player = MCTSPlayer(net=net, mcts_config=mcts_config, temperature=0.0, name="my_player")
print(f"Player: {player.name}")
print(f"Device: {net.device}")
print(f"MCTS sims: {mcts_config.num_simulations}")

## 2. Save a player to disk

In [ ]:
player.save("../models/my_player")

# What got saved?
import json
from pathlib import Path

saved_dir = Path("../models/my_player")
print("Saved files:", [f.name for f in saved_dir.iterdir()])
print()

config = json.loads((saved_dir / "player.json").read_text())
print("player.json:")
print(json.dumps(config, indent=2))

## 3. Load a player from disk

No need to know the net class — it's stored in `player.json` and imported automatically.

In [ ]:
loaded_player = MCTSPlayer.load("../models/my_player")

print(f"Player: {loaded_player.name}")
print(f"Temperature: {loaded_player.temperature}")
print(f"MCTS sims: {loaded_player.mcts.mcts_config.num_simulations}")
print(f"Net device: {loaded_player.mcts.net.device}")
print(f"Net type: {type(loaded_player.mcts.net).__name__}")

## 4. Verify: original vs loaded play the same

Both should produce identical moves from the same position (temperature=0 → deterministic).

In [ ]:
game = BridgitGame(board_config)

action_original = player.get_action(game)
action_loaded = loaded_player.get_action(game)

print(f"Original player action: {action_original}")
print(f"Loaded player action:   {action_loaded}")
print(f"Match: {action_original == action_loaded}")

## 5. Load a player from a training run

`from_training_iteration` loads the neural net weights and MCTS config from a training run directory.

In [ ]:
from pathlib import Path

training_dir = Path("../trainings")
for run_dir in sorted(training_dir.glob("run_*")):
    iterations = sorted(run_dir.glob("iteration_*"))
    has_config = (run_dir / "run_config.json").exists()
    print(f"{run_dir.name}: {len(iterations)} iterations, config={'yes' if has_config else 'no'}")
    for it in iterations:
        files = [f.name for f in it.iterdir()]
        print(f"  {it.name}: {files}")

In [ ]:
RUN_DIR = "../trainings/run_2026-03-28_203716"

# Load from a specific iteration
trained_player = MCTSPlayer.from_training_iteration(
    f"{RUN_DIR}/iteration_005",
    net_class=BridgitNet,
)
print(f"Loaded: {trained_player.name}")
print(f"MCTS sims: {trained_player.mcts.mcts_config.num_simulations}")

# Or load the latest iteration from a run directory
latest_player = MCTSPlayer.from_training_iteration(
    RUN_DIR,
    net_class=BridgitNet,
)
print(f"\nLatest: {latest_player.name}")

# Override MCTS config for stronger play
strong_player = MCTSPlayer.from_training_iteration(
    RUN_DIR,
    net_class=BridgitNet,
    mcts_config=MCTSConfig(num_simulations=1000, num_parallel_leaves=50),
)
print(f"\nStrong: {strong_player.name}, sims={strong_player.mcts.mcts_config.num_simulations}")

## 6. Pit trained vs untrained player

In [ ]:
# Trained (from training run) vs untrained (random weights)
untrained_net = BridgitNet(board_config=board_config, net_config=NeuralNetConfig(num_channels=32, num_res_blocks=4))

results = batched_arena(
    net_a=trained_player.mcts.net,
    net_b=untrained_net,
    mcts_config=trained_player.mcts.mcts_config,
    game_factory=lambda: BridgitGame(board_config),
    num_games=40,
    batch_size=40,
    name_a="trained",
    name_b="untrained",
    swap_players=True,
    verbose=True,
)

print(f"\nResults: {results.scores}")